# Z potential: committor from many short trajectories

The 2D Brownian Z potential of the staple run `sim_istarz_br_g10` (toytis `sjoelbak_angle`, Lx = 0.3, 45° ridge; overdamped, γ = 10, kT = 0.05, dt = 0.002; A: y ≤ 0.05, B: y ≥ 0.85, y = order parameter).

**Part 1** (original `optimalrcs-cupy`): take a saved first-try committor (any `committor.npz` with the frames and their q), pick start frames evenly over bins of its natural coordinate q_n (or of q), run many short trajectories of the real dynamics from them, and fit NPNEq (`CommittorNE`) on those trajectories. The new trajectories are plain dynamics (no path selection, no weights), so the fit is unbiased whatever the start frames are; the bins only decide where the data are. The result can be saved in the same format and used as the next first try.

**Part 2** (Elias' TIS version of the library, restart the kernel first): read the staple paths and fit the committor on them; it saves the `committor.npz` that part 1 can start from.

## Part 1: short trajectories from a first-try committor

### Imports (original optimalrcs-cupy)

In [ ]:
import os
import sys
import time

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

plt.rc('font', size=14)          # controls default text sizes
plt.rc('axes', titlesize=16)     # fontsize of the axes title
plt.rc('axes', labelsize=16)     # fontsize of the x and y labels
plt.rc('xtick', labelsize=14)    # fontsize of the tick labels
plt.rc('ytick', labelsize=14)    # fontsize of the tick labels
plt.rc('legend', fontsize=12)    # legend fontsize

ORIG_LIB = '/home/staff7/fbsskr/Notebooks/optimalrcs-cupy/'      # the original optimalrcs-cupy
sys.path.append(ORIG_LIB)
import optimalrcs, metrics, plots

FIG = 'figures/nb_zpot_short_traj'                               # figures and data of this notebook
os.makedirs(FIG, exist_ok=True)

def show(call, name):
    """Run a plotting call of the library (which calls plt.show itself), save the figure, then show it."""
    plt_show = plt.show
    plt.show = lambda *a, **k: None
    try:
        call()
    finally:
        plt.show = plt_show
    plt.gcf().savefig(f'{FIG}/{name}.png', dpi=100, bbox_inches='tight')
    plt.show()

### The Z potential and its dynamics

`force(y, x)` is the force of toytis' `RectangularGridWithAngledBarrierPotential(Lx=0.3, angle=45)` term by term (walls, the tilted cosine ridge, the cosine wells of A and B, the slope between them and gravity), on GPU arrays. `potential(y, x)` is its effective potential (the class' `potential` plus gravity × y, so that force = −∇V). If toytis can be imported, the force is checked against the class.

The dynamics is the toytis overdamped update: y ← y + dt/γ F_y + σ ξ, the same for x, σ = √(2 kT dt / γ), absorption checked every step.

In [ ]:
kT, gamma_md, dt = 0.05, 10.0, 0.002      # dynamics of sim_istarz_br_g10
lambda_A, lambda_B = 0.05, 0.85            # A: y <= lambda_A, B: y >= lambda_B

# parameters of the potential (toytis defaults, Lx = 0.3, angle = 45 degrees)
Lx, Ly, k_wall = 0.3, 0.9, 2000.0
A_ridge, L_ridge = 0.3, 0.2                # ridge height and length scale (half-width L/4)
slope_r = -np.tan(np.radians(45.0))        # the ridge line y = slope_r * x + b passes through (x, y) = (Lx/2, 0.45)
b_ridge = 0.45 - slope_r * Lx / 2
wells = [(0.025, -0.15, 0.2), (0.875, -0.15, 0.2)]   # (centre y, amplitude, length scale) of the wells of A and B
slope_start, slope_end, slope_strength = 0.1, 0.825, 0.2
gravity = 0.2

def force(y, x, xp=cp):
    """Force (F_y, F_x) of the Z potential at arrays (y, x)."""
    fx = xp.where(x < 0, -k_wall * x, xp.where(x > Lx, -k_wall * (x - Lx), 0.0))        # walls
    fy = xp.where(y < 0, -k_wall * y, xp.where(y > Ly, -k_wall * (y - Ly), 0.0))
    norm = np.sqrt(slope_r ** 2 + 1)                                                     # ridge
    D = xp.abs(slope_r * x - y + b_ridge) / norm
    dV_dD = xp.where(D < L_ridge / 4, -2 * np.pi * A_ridge / L_ridge * xp.sin(2 * np.pi * D / L_ridge), 0.0)
    side = xp.sign(y - slope_r * x - b_ridge)
    fx = fx + side * dV_dD * slope_r / norm
    fy = fy - side * dV_dD / norm
    for yc, amp, ell in wells:                                                           # wells of A and B
        Dw = xp.abs(y - yc)
        fy = fy + xp.where(Dw < ell / 4, amp * 2 * np.pi / ell * xp.sin(2 * np.pi * Dw / ell) * xp.sign(y - yc), 0.0)
    inside = (y >= slope_start) & (y <= slope_end)                                       # slope and gravity
    fy = fy + xp.where(inside, -slope_strength / (slope_end - slope_start), 0.0) - gravity
    return fy, fx

def potential(y, x, xp=np):
    """Effective potential of the Z potential (force = -grad V)."""
    v = xp.where(x < 0, 0.5 * k_wall * x ** 2, xp.where(x > Lx, 0.5 * k_wall * (x - Lx) ** 2, 0.0))
    v = v + xp.where(y < 0, 0.5 * k_wall * y ** 2, xp.where(y > Ly, 0.5 * k_wall * (y - Ly) ** 2, 0.0))
    D = xp.abs(slope_r * x - y + b_ridge) / np.sqrt(slope_r ** 2 + 1)
    v = v + xp.where(D < L_ridge / 4, A_ridge * xp.cos(2 * np.pi * D / L_ridge), 0.0)
    for yc, amp, ell in wells:
        Dw = xp.abs(y - yc)
        v = v + xp.where(Dw < ell / 4, amp * xp.cos(2 * np.pi * Dw / ell), 0.0)
    v = v + xp.where(y < slope_start, 0.0, xp.where(y <= slope_end, slope_strength * (y - slope_start) / (slope_end - slope_start), slope_strength))
    return v + gravity * y

# check against the toytis class, if it can be imported
TOYTIS = '/home/staff7/fbsskr/Notebooks/Students/elias/toytis'
try:
    sys.path.insert(0, TOYTIS)
    from potentials.sjoelbak_angle import RectangularGridWithAngledBarrierPotential
    pot = RectangularGridWithAngledBarrierPotential(Lx=0.3, angle=45)
    pts = np.random.default_rng(1).uniform([-0.02, -0.02], [0.92, 0.32], size=(300, 2))
    f_here = np.column_stack([f.get() for f in force(cp.asarray(pts[:, 0]), cp.asarray(pts[:, 1]))])
    f_class = np.array([pot.potential_and_force((p, np.zeros(2)))[1] for p in pts])
    v_here = potential(pts[:, 0], pts[:, 1])
    v_class = np.array([pot.effective_potential((p, np.zeros(2))) for p in pts])
    print('force and potential agree with the toytis class:', np.allclose(f_here, f_class, atol=1e-9), np.allclose(v_here, v_class, atol=1e-9))
except ImportError:
    print('toytis not found: force and potential not checked against the class')

gy, gx = np.linspace(-0.02, 0.92, 189), np.linspace(-0.02, 0.32, 69)
V = potential(*np.meshgrid(gy, gx, indexing='ij'))
V_levels = np.arange(0, 12.5, 1.0)
def draw_potential(ax):
    ax.contour(gy, gx, ((V - V.min()) / kT).T, levels=V_levels, colors='0.6', linewidths=0.5)
    for lam in (lambda_A, lambda_B):
        ax.axvline(lam, color='k', lw=1)
    ax.set(xlabel='y (order parameter)', ylabel='x', xlim=(-0.02, 0.92), ylim=(-0.02, 0.32))

### The first-try committor

Any `committor.npz` with the frames and their committor: `X_all` (columns y, x), `r` (or `q_fit`), and for q_n also `i_traj`, `t_traj` (and `path_weights` if the frames are path data). Part 2 saves such a file; so do `load_tis_data_cupy.py` and the staple notebooks.

`BIN_ON = 'qn'`: start frames evenly over bins of the natural coordinate q_n of the first try (`plots.transform_q2qn`): equal steps in q_n are steps of equal difficulty for the dynamics, so the starts are spread over the transition region instead of piling up where the paths spend their time. `BIN_ON = 'q'`: bins of q itself.

In [ ]:
FIRST_TRY_NPZ = '/home/staff7/fbsskr/Notebooks/Students/elias/optimalrcs/examples/figures/istarz_br_g10/committor.npz'
BIN_ON = 'qn'              # 'qn' or 'q'
N_BINS = 30

d = np.load(FIRST_TRY_NPZ)
q_first = (d['r'] if 'r' in d.files else d['q_fit']).astype(float)
y_first, x_first = d['X_all'][:, 0].astype(float), d['X_all'][:, 1].astype(float)
free_first = (y_first > lambda_A) & (y_first < lambda_B)
print(f'{FIRST_TRY_NPZ}: {len(q_first):,} frames, {free_first.sum():,} outside A and B')

if BIN_ON == 'qn':
    w_first = cp.asarray(d['path_weights']) if 'path_weights' in d.files else None
    qn_first = plots.transform_q2qn(cp.asarray(q_first), i_traj=cp.asarray(d['i_traj']), t_traj=cp.asarray(d['t_traj']),
                                    w_traj=w_first, dt_sim=dt)
    coord = qn_first / qn_first.max()
else:
    coord = q_first
bin_first = np.clip((coord * N_BINS).astype(int), 0, N_BINS - 1)
n_per_bin = np.bincount(bin_first[free_first], minlength=N_BINS)
print(f'frames per bin of {BIN_ON}:', n_per_bin)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 5))
show_frames = np.random.default_rng(0).choice(np.flatnonzero(free_first), size=min(200_000, free_first.sum()), replace=False)
sc = ax1.scatter(y_first[show_frames], x_first[show_frames], c=q_first[show_frames], s=1, cmap='coolwarm', vmin=0, vmax=1)
draw_potential(ax1)
fig.colorbar(sc, ax=ax1, label='first-try q')
ax1.set_title('first-try committor at its frames')
ax2.bar(np.arange(N_BINS), n_per_bin)
ax2.set(yscale='log', xlabel=f'bin of {BIN_ON}', ylabel='frames', title='frames per bin (start frames are drawn evenly over the bins)')
fig.tight_layout()
fig.savefig(f'{FIG}/first_try.png', dpi=100, bbox_inches='tight')
plt.show()

### Many short trajectories

`N_TRAJ` start frames, `N_TRAJ / N_BINS` per bin (fewer if a bin has fewer frames), one trajectory of at most `L` steps from each, stopped when it reaches A or B. All trajectories run at once on the GPU; a stopped trajectory is frozen and its later frames are not used.

In [ ]:
N_TRAJ = 20_000
L = 300                    # steps per trajectory
SEED = 0

rng = np.random.default_rng(SEED)
start = np.concatenate([rng.choice(np.flatnonzero(free_first & (bin_first == b)), size=min(N_TRAJ // N_BINS, n_per_bin[b]), replace=False)
                        for b in range(N_BINS)])
n_traj = len(start)

sigma = np.sqrt(2 * kT * dt / gamma_md)
cp.random.seed(SEED)
y_now, x_now = cp.asarray(y_first[start]), cp.asarray(x_first[start])
Y_frames, X_frames = cp.zeros((L + 1, n_traj)), cp.zeros((L + 1, n_traj))      # [step, trajectory]
Y_frames[0], X_frames[0] = y_now, x_now
last = cp.full(n_traj, L)                       # last frame of each trajectory
running = cp.ones(n_traj, bool)
t0 = time.time()
for step in range(1, L + 1):
    fy, fx = force(y_now, x_now)
    y_now = cp.where(running, y_now + dt / gamma_md * fy + sigma * cp.random.standard_normal(n_traj), y_now)
    x_now = cp.where(running, x_now + dt / gamma_md * fx + sigma * cp.random.standard_normal(n_traj), x_now)
    Y_frames[step], X_frames[step] = y_now, x_now
    absorbed = running & ((y_now <= lambda_A) | (y_now >= lambda_B))
    last = cp.where(absorbed, step, last)
    running = running & ~absorbed
last = last.get()
print(f'{n_traj} trajectories of up to {L} steps in {time.time() - t0:.1f} s; '
      f'{np.mean(last < L):.0%} reached A or B')

# one segment per trajectory: frames 0 .. last
keep = np.arange(L + 1)[None, :] <= last[:, None]                  # [trajectory, step]
y = Y_frames.get().T[keep]
x = X_frames.get().T[keep]
i_traj = np.repeat(np.arange(n_traj), last + 1)
t_traj = np.concatenate([np.arange(n + 1) for n in last]).astype(float)
first_frame = np.r_[0, np.cumsum(last + 1)[:-1]]                  # index of each start frame
del Y_frames, X_frames
print(f'{len(y):,} frames; {np.sum(y <= lambda_A)} in A, {np.sum(y >= lambda_B)} in B')
np.savez(f'{FIG}/short_traj_{BIN_ON}_n{n_traj}_L{L}.npz', y=y, x=x, i_traj=i_traj, t_traj=t_traj, start=start)

fig, ax = plt.subplots(figsize=(11, 5))
draw_potential(ax)
ax.scatter(y[first_frame], x[first_frame], s=2, c='C0', label='start frames')
for j in np.random.default_rng(1).choice(n_traj, 30, replace=False):
    ax.plot(y[i_traj == j], x[i_traj == j], lw=0.6)
ax.legend(loc='upper left')
ax.set_title(f'{n_traj} start frames over {N_BINS} bins of {BIN_ON}, and 30 of the trajectories')
fig.savefig(f'{FIG}/trajectories.png', dpi=100, bbox_inches='tight')
plt.show()

### NPNEq fit on the short trajectories

`CommittorNE` with one segment per trajectory. The trajectories that reach A or B are treated as staying there (`set_fixed_traj_length_trap`; this only affects the Z_q plots, not the fit). CVs: y and x, one drawn at random every iteration. A warm-up with a very wide envelope, then the usual envelope 0.01.

In [ ]:
boundary0, boundary1 = y <= lambda_A, y >= lambda_B

q = optimalrcs.CommittorNE(boundary0=boundary0, boundary1=boundary1, i_traj=i_traj, t_traj=t_traj)
q.set_fixed_traj_length_trap(boundary0 | boundary1, 2 ** 16)

cvs = [cp.asarray(y), cp.asarray(x)]
def comp_y():
    return cvs[np.random.randint(len(cvs))]

metrics_print = ('iter', 'delta_r2', 'max_sd_zq', 'delta_x', 'time_elapsed')
np.random.seed(0)
cp.random.seed(0)

optimalrcs.envelope_scale = 1e7          # warm-up
q.fit_transform(comp_y, gamma=0, ny=6, max_iter=1000, print_step=500, metrics_print=metrics_print,
                delta_r2_max_change_allowed=10000)
optimalrcs.envelope_scale = 0.01         # main fit
q.fit_transform(comp_y, gamma=0, ny=6, max_iter=20000, print_step=1000, metrics_print=metrics_print)

In [ ]:
show(lambda: q.plots_metrics(['delta_r2', 'max_sd_zq', 'delta_x']), 'fit_metrics')
show(lambda: q.plots_feps(delta_t_sim=dt), 'fit_feps')
show(lambda: q.plots_obs_pred(), 'fit_obs_pred')
show(lambda: q.plots_obs_pred(log_scale=True), 'fit_obs_pred_log')

### Comparison with the exact committor and the first try

The exact committor of this dynamics (finite-volume solution of the backward equation, `figures/z_brownian/committor_reference.py`; checked by shooting to absorption at 200 path frames, see the notes §13.1), if the file is there. Engine shooting at four points on the plateau beyond the ridge. The new fit at the start frames (each is a frame of the first-try data too) against the first try.

In [ ]:
from scipy.interpolate import RegularGridInterpolator
REFERENCE_NPZ = '/home/staff7/fbsskr/Notebooks/Students/elias/optimalrcs/examples/figures/z_brownian/committor_reference_h0.001.npz'

r = q.r_traj.get()
free = ~(boundary0 | boundary1)
ey, ex = np.linspace(lambda_A, lambda_B, 81), np.linspace(-0.01, 0.31, 33)
counts = np.histogram2d(y[free], x[free], bins=(ey, ex))[0]
def bin_mean(values):
    s = np.histogram2d(y[free], x[free], bins=(ey, ex), weights=values[free])[0]
    return np.where(counts >= 20, s / np.maximum(counts, 1), np.nan)
Q_fit = bin_mean(r)

q_exact = None
if os.path.exists(REFERENCE_NPZ):
    ref = np.load(REFERENCE_NPZ)
    f_ref = RegularGridInterpolator((ref['y'], ref['x']), ref['q'], bounds_error=False, fill_value=None)
    q_exact = np.clip(f_ref(np.column_stack([y, x])), 0, 1)
    Q_exact = bin_mean(q_exact)
    transition = (Q_exact > 0.05) & (Q_exact < 0.95)
    print(f'RMS(fit - exact) per (y, x) bin, 0.05 < q_exact < 0.95: {np.sqrt(np.nanmean((Q_fit - Q_exact)[transition] ** 2)):.4f}')
else:
    print('no reference file: no exact committor')

# engine shooting at four plateau points (1200 trajectories each, run until A or B)
shooting = [((0.50, 0.25), 0.639), ((0.55, 0.10), 0.664), ((0.62, 0.15), 0.698), ((0.70, 0.15), 0.747)]
print('point (y, x)    shooting   new fit   first try')
for (py, px), q_shoot in shooting:
    near = free & ((y - py) ** 2 + (x - px) ** 2 < 0.01 ** 2)
    near_first = free_first & ((y_first - py) ** 2 + (x_first - px) ** 2 < 0.01 ** 2)
    print(f'({py:.2f}, {px:.2f})     {q_shoot:.3f}     {r[near].mean():.3f}     {q_first[near_first].mean():.3f}')

at_start = {'new fit': r[first_frame], 'first try': q_first[start]}
if q_exact is not None:
    at_start['exact'] = q_exact[first_frame]
for name in [n for n in at_start if n != 'new fit']:
    diff = at_start['new fit'] - at_start[name]
    print(f'at the {n_traj} start frames: new fit - {name}: mean {diff.mean():+.4f}, RMS {np.sqrt(np.mean(diff ** 2)):.4f}')

panels = [(Q_fit, 'new fit', 'coolwarm', 0, 1)]
if q_exact is not None:
    panels += [(Q_exact, 'exact', 'coolwarm', 0, 1), (Q_fit - Q_exact, 'new fit - exact', 'PuOr', -0.15, 0.15)]
fig, axes = plt.subplots(1, len(panels), figsize=(7 * len(panels), 4.5), squeeze=False)
for ax, (M, title, cmap, vmin, vmax) in zip(axes[0], panels):
    im = ax.pcolormesh(ey, ex, M.T, cmap=cmap, vmin=vmin, vmax=vmax)
    ax.contour(gy, gx, ((V - V.min()) / kT).T, levels=V_levels, colors='0.6', linewidths=0.5)
    ax.set(title=title, xlabel='y', ylabel='x', xlim=(lambda_A, lambda_B), ylim=(-0.01, 0.31))
    fig.colorbar(im, ax=ax)
fig.tight_layout()
fig.savefig(f'{FIG}/maps.png', dpi=100, bbox_inches='tight')
plt.show()

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot(at_start['first try'], at_start['new fit'], '.', ms=2, alpha=0.4, label='first try')
if q_exact is not None:
    ax.plot(at_start['exact'], at_start['new fit'], '.', ms=2, alpha=0.4, label='exact')
ax.plot([0, 1], [0, 1], 'k--')
ax.set(xlabel='q', ylabel='new fit', title='q at the start frames')
ax.legend(markerscale=6)
fig.savefig(f'{FIG}/at_start_frames.png', dpi=100, bbox_inches='tight')
plt.show()

### Save

In the format the first-try cell reads (`X_all`, `r`, `i_traj`, `t_traj`), so the next round can start from this fit (set `FIRST_TRY_NPZ` to it).

In [ ]:
OUT_NPZ = f'{FIG}/committor_short_traj_{BIN_ON}_n{n_traj}_L{L}.npz'
np.savez(OUT_NPZ, r=r, X_all=np.column_stack([y, x]), i_traj=i_traj, t_traj=t_traj,
         boundary0=boundary0, boundary1=boundary1, start=start)
print('saved', OUT_NPZ)

## Part 2: staple paths and their committor (Elias' TIS version)

This part needs the modified library (`optimalrcs-cupy` of the `partial_tis` branch: path weights, independence groups, stopped Z_q, lazy basis) and `examples/tis_data.py`. Both use the same module names as the original library, so **restart the kernel before running it**; it does not need part 1.

The paths are read with `tis_data.prepare_data`: every accepted path is cut at its first frame past its ensemble's interface M, its mirrored piece is added (`--reversed`), and every piece carries its path's rejection weight. The first 40% of the MC cycles of every ensemble are discarded (`--burn-in-fraction`), and `--paths-per-ensemble` keeps that many paths evenly spaced over the rest of the chain.

In [ ]:
import argparse
import os
import sys

import numpy as np
import cupy as cp
import matplotlib.pyplot as plt

REPO = '/home/staff7/fbsskr/Notebooks/Students/elias/optimalrcs'     # partial_tis checkout
sys.path.insert(0, f'{REPO}/optimalrcs-cupy/')
sys.path.insert(0, f'{REPO}/examples/')
import optimalrcs, tis_data

FIG = 'figures/nb_zpot_short_traj'
os.makedirs(FIG, exist_ok=True)

In [ ]:
TIS_DIR = '/home/staff7/fbsskr/Notebooks/Students/elias/data/simulations/sim_istarz_br_g10'
PATHS_PER_ENSEMBLE = 1500

parser = argparse.ArgumentParser()
tis_data.add_data_arguments(parser)
args = parser.parse_args([TIS_DIR, '--reversed', '--weights', 'rejections', '--burn-in-fraction', '0.4',
                          '--paths-per-ensemble', str(PATHS_PER_ENSEMBLE)])
data = tis_data.prepare_data(args)
y_path, x_path = data.X_all[:, 0], data.X_all[:, 1]
print(f'{len(y_path):,} frames in {len(np.unique(data.i_traj))} pieces from {len(np.unique(data.group_traj))} paths')

In [ ]:
q_path = optimalrcs.CommittorNE(boundary0=data.boundary0, boundary1=data.boundary1,
                                i_traj=data.i_traj, t_traj=data.t_traj,
                                path_weights=data.path_weights,      # MC weight of each frame's path
                                group_traj=data.group_traj)          # forward and mirrored piece of a path: one unit
q_path.set_fixed_traj_length_trap(data.boundary0 | data.boundary1, 2 ** 16)

cvs_path = [cp.asarray(y_path), cp.asarray(x_path)]
def comp_y_path():
    return cvs_path[np.random.randint(len(cvs_path))]

fit_kw = dict(ny=6, lazy_basis=True, save_min_metric='max_z_zq_stopped',
              metrics_print=('iter', 'delta_r2', 'max_z_zq_stopped', 'max_sd_zq_stopped', 'delta_x', 'time_elapsed'))
np.random.seed(0)
cp.random.seed(0)

optimalrcs.envelope_scale = 1           # warm-up
q_path.fit_transform(comp_y_path, gamma=0.1, max_iter=1000, print_step=500, **fit_kw)
optimalrcs.envelope_scale = 0.01        # main fit
q_path.fit_transform(comp_y_path, gamma=0.1, max_iter=15000, print_step=1000, **fit_kw)

q_path.plots_metrics(['delta_r2', 'max_z_zq_stopped', 'delta_x'])
q_path.plots_tis()                        # stopped Z_q: the test for path data
q_path.plots_feps(delta_t_sim=0.002)      # with the path weights

In [ ]:
# save in the format part 1 reads (FIRST_TRY_NPZ)
OUT_NPZ = f'{FIG}/committor_staple_p{PATHS_PER_ENSEMBLE}.npz'
np.savez(OUT_NPZ, r=q_path.r_traj.get(), X_all=data.X_all, i_traj=data.i_traj, t_traj=data.t_traj,
         path_weights=data.path_weights, boundary0=data.boundary0, boundary1=data.boundary1)
print('saved', OUT_NPZ)